<a href="https://colab.research.google.com/github/kumarsuman-dev/GoogleColab_Repo/blob/main/Hindi_to_English_Speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hindi Speech → English Speech

In [2]:
!pip install -q transformers sentencepiece sacremoses gTTS librosa soundfile

In [37]:
def translate(hindi_text):
    tok.src_lang = "hin_Deva"

    inputs = tok(
        hindi_text,
        return_tensors="pt"
    ).to(mt.device)

    tokens_list = tok.convert_ids_to_tokens(
        inputs["input_ids"][0].tolist()
    )

    print("Hindi Tokens:", tokens_list)

    with torch.no_grad():
        # Encoder vector representation
        vec = mt.get_encoder()(**inputs).last_hidden_state
        print("Vector shape:", tuple(vec.shape))

        # Hindi to English
        english_id = tok.convert_tokens_to_ids("eng_Latn")

        out = mt.generate(
            **inputs,
            forced_bos_token_id=english_id,
            max_new_tokens=128,
            num_beams=4
        )

    return tok.decode(out[0], skip_special_tokens=True)

In [42]:
def speech_to_text(signal, sr):
    # Normalize audio safely
    signal = np.asarray(signal, dtype=np.float32)
    signal = signal / (np.max(np.abs(signal)) + 1e-8)

    result = asr(
        {"raw": signal, "sampling_rate": sr},
        chunk_length_s=15,
        generate_kwargs={
            "language": "hindi",
            "task": "transcribe"
        }
    )

    return result["text"].strip()

In [43]:
def hindi_to_english_voice():
    signal, sr = record()

    if signal is None:
        return

    print("Hindi Audio Vector Shape:", signal.shape)

    hindi = speech_to_text(signal, sr)
    print("\nHindi Text:", hindi)

    english = translate(hindi)
    print("\nEnglish Translation:", english)

    print("\nEnglish Speech Output:")
    display(text_to_speech(english))

In [60]:
# Hindi Voice Input -> English Voice Output

hindi_to_english_voice()

<IPython.core.display.Javascript object>


Awaaz level (0.01 se upar hona chahiye): 0.5322


[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Hindi Audio Vector Shape: (58264,)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Hindi Text: मेरा नाम सूमन है
Hindi Tokens: ['hin_Deva', '▁मेरा', '▁नाम', '▁सू', 'मन', '▁है', '</s>']
Vector shape: (1, 7, 1024)

English Translation: My name is Suman.

English Speech Output:
